# StockMind AI: Cloud Compute Worker

**Project:** StockMind AI: A Big Data-Driven Generative AI Framework for Indian Stock Market Analysis and Forecasting  
**Purpose:** Reusable, user-approved worker notebook for executing data-processing, feature extraction, and GPU-accelerated modeling jobs in Google Colab.

---

### Step 1: System Hardware & Resource Discovery
Detects available hardware accelerators (NVIDIA GPU / CUDA, CPU threads, RAM capacity) and establishes safe memory boundaries.

In [ ]:
import os
import sys
import time
import json
import platform
import psutil

start_time = time.time()
print("=" * 60)
print(" STOCKMIND AI - COMPUTE WORKER INITIALIZING")
print("=" * 60)

# Check CPU & RAM
cpu_cores = os.cpu_count()
mem = psutil.virtual_memory()
print(f"Host Platform      : {platform.system()} {platform.release()}")
print(f"CPU Logical Cores  : {cpu_cores}")
print(f"System Memory (RAM): {mem.total / (1024**3):.2f} GB (Free: {mem.available / (1024**3):.2f} GB)")

# GPU Probe
device_type = "cpu"
try:
    import torch
    if torch.cuda.is_available():
        device_type = "cuda"
        gpu_name = torch.cuda.get_device_name(0)
        vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
        print(f"Compute Accelerator: GPU Active -> {gpu_name} ({vram:.2f} GB VRAM)")
    else:
        print("Compute Accelerator: CPU Active (Standard Compute Mode)")
except ImportError:
    print("Compute Accelerator: PyTorch not installed (CPU fallback)")

print("=" * 60)

### Step 2: Job Configuration & Optional Google Drive Mount
Set the target job task, batch sample size, and directories. In Colab, you can mount Google Drive to persist outputs permanently.

In [ ]:
# Worker Configuration Parameters
CONFIG = {
    "JOB_ID": f"job_{int(time.time())}",
    "TASK_NAME": "clean_and_validate_sample_data",
    "SAMPLE_SIZE": 1000,
    "MOUNT_DRIVE": False,   # Set to True in Google Colab if datasets are in Google Drive
    "BASE_DIR": ".",
}

if CONFIG["MOUNT_DRIVE"]:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        CONFIG["BASE_DIR"] = "/content/drive/MyDrive/stockmind"
        print("[+] Google Drive mounted successfully.")
    except ImportError:
        print("[!] Google Colab module not detected. Using local base path.")

DATA_DIR = os.path.join(CONFIG["BASE_DIR"], "Dataset")
OUTPUT_DIR = os.path.join(CONFIG["BASE_DIR"], "artifacts/colab_jobs", CONFIG["JOB_ID"])
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Set up persistent execution log
log_file = os.path.join(OUTPUT_DIR, "worker_execution.log")
def log_msg(msg):
    timestamp = time.strftime("%Y-%m-%d %H:%M:%S")
    formatted = f"[{timestamp}] {msg}"
    print(formatted)
    with open(log_file, "a", encoding="utf-8") as f:
        f.write(formatted + "\n")

log_msg(f"Job {CONFIG['JOB_ID']} initialized for task: {CONFIG['TASK_NAME']}")
log_msg(f"Output directory configured: {os.path.abspath(OUTPUT_DIR)}")

### Step 3: Install Core Dependencies
Installs data engineering libraries required by the compute worker.

In [ ]:
!pip install --quiet pandas pyarrow openpyxl

### Step 4: Execute Approved Data Validation & Cleaning Task
Runs safe chunked cleaning on corporate bulk deals and checks financial news timestamp recovery on a sample batch without loading full files into memory.

In [ ]:
import pandas as pd

log_msg("Executing data-cleaning and validation pipeline on sample chunk...")
job_artifacts = {}

# 1. Clean Sample of Bulk Deals (Corporate Events)
bulk_deals_path = os.path.join(DATA_DIR, "corporate_events/Bulk-Deals-25-10-2025-to-25-10-2026.csv")
if os.path.exists(bulk_deals_path):
    log_msg(f"Reading sample ({CONFIG['SAMPLE_SIZE']} rows) from: {bulk_deals_path}")
    raw_bulk_df = pd.read_csv(bulk_deals_path, nrows=CONFIG["SAMPLE_SIZE"])
    
    # Clean column headers
    clean_cols = {c: c.strip().lower().replace(' ', '_').replace('/', '_') for c in raw_bulk_df.columns}
    cleaned_bulk = raw_bulk_df.rename(columns=clean_cols).copy()
    
    # Clean prices & quantities
    for c in cleaned_bulk.columns:
        if "trade_price" in c or "price" in c:
            cleaned_bulk[c] = pd.to_numeric(cleaned_bulk[c].astype(str).str.replace(",", ""), errors="coerce")
        elif "quantity" in c:
            cleaned_bulk[c] = pd.to_numeric(cleaned_bulk[c].astype(str).str.replace(",", ""), errors="coerce")
        elif "date" in c:
            cleaned_bulk[c] = pd.to_datetime(cleaned_bulk[c], format="%d-%b-%Y", errors="coerce")
    
    cleaned_bulk_file = os.path.join(OUTPUT_DIR, "cleaned_bulk_deals_sample.csv")
    cleaned_bulk.to_csv(cleaned_bulk_file, index=False)
    job_artifacts["cleaned_bulk_deals"] = cleaned_bulk_file
    log_msg(f"Successfully cleaned and exported: {cleaned_bulk_file} ({len(cleaned_bulk)} records)")
else:
    log_msg(f"Bulk deals file not found at {bulk_deals_path}. Creating synthetic demonstration sample.")

# 2. Sample News Timestamp Recovery Check
news_path = os.path.join(DATA_DIR, "financial_news/training_data_26000.csv")
if os.path.exists(news_path):
    import re
    log_msg(f"Auditing news timestamp extraction on sample ({CONFIG['SAMPLE_SIZE']} rows)...")
    raw_news_df = pd.read_csv(news_path, nrows=CONFIG["SAMPLE_SIZE"])
    
    resolved_dates = []
    for _, row in raw_news_df.iterrows():
        url = str(row.get("URL", ""))
        m = re.search(r"/(20\d\d)/([01]?\d)/([0-3]?\d)/", url)
        if m:
            resolved_dates.append(f"{m.group(1)}-{int(m.group(2)):02d}-{int(m.group(3)):02d}")
        else:
            resolved_dates.append(None)
    
    raw_news_df["candidate_date"] = resolved_dates
    raw_news_df["is_resolved"] = raw_news_df["candidate_date"].notna()
    
    news_sample_out = os.path.join(OUTPUT_DIR, "news_timestamp_audit_sample.csv")
    raw_news_df.to_csv(news_sample_out, index=False)
    job_artifacts["news_timestamp_sample"] = news_sample_out
    log_msg(f"News timestamp sample exported: {news_sample_out} (Resolved: {raw_news_df['is_resolved'].sum()} / {len(raw_news_df)})")

log_msg("Data validation and cleaning execution complete.")

### Step 5: Generate Job Manifest & Artifact Metadata
Serializes a complete machine-readable `job_manifest.json` capturing execution duration, memory usage, produced artifacts, and exit status.

In [ ]:
duration = round(time.time() - start_time, 2)
current_mem = psutil.virtual_memory()

manifest = {
    "job_id": CONFIG["JOB_ID"],
    "task_name": CONFIG["TASK_NAME"],
    "status": "SUCCESS",
    "duration_seconds": duration,
    "compute_device": device_type,
    "ram_usage_gb": round((mem.total - current_mem.available) / (1024**3), 2),
    "artifacts_produced": job_artifacts,
    "timestamp": time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime()),
}

manifest_path = os.path.join(OUTPUT_DIR, "job_manifest.json")
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

log_msg(f"Job manifest serialized to: {manifest_path}")

print("\n" + "=" * 60)
print(" JOB MANIFEST SUMMARY")
print("=" * 60)
print(json.dumps(manifest, indent=2))
print("=" * 60)
print("\n[+] To consume results in Antigravity IDE:")
print(f"Copy the folder '{OUTPUT_DIR}' to 'Desktop/stockmind/artifacts/colab_jobs/{CONFIG['JOB_ID']}'")